In [49]:
import os
import mne
import numpy as np
import pywt
import matplotlib.pyplot as plt
from tqdm import tqdm

# Игнорировать избыточные предупреждения MNE
mne.set_log_level('WARNING')

# Конфигурация путей (пожалуйста, измените в соответствии с вашим фактическим путем)
DATA_DIR = './data/chb08'
OUT_DIR_SEIZURE = './dataset/seizure'
OUT_DIR_NORMAL = './dataset/normal'

os.makedirs(OUT_DIR_SEIZURE, exist_ok=True)
os.makedirs(OUT_DIR_NORMAL, exist_ok=True)

# Настройки частоты дискретизации и окна
SFREQ = 256
WINDOW_SEC = 4
WINDOW_SAMPLES = SFREQ * WINDOW_SEC

# Информация, извлеченная из chb08-summary.txt
# Формат: (имя файла, начало приступа в секундах, конец приступа в секундах)
seizure_records = [
    ('chb08_02.edf', 2670, 2841),
    ('chb08_05.edf', 2856, 3046),
    ('chb08_11.edf', 2988, 3122),
    ('chb08_13.edf', 2417, 2577),
    ('chb08_21.edf', 2083, 2347)
]
# Справочные файлы без эпилептических приступов
normal_records = ['chb08_03.edf', 'chb08_04.edf', 'chb08_10.edf']

# Изменение шага в функции extract_segments
def extract_segments(raw, start_sec, end_sec, max_segments):
    data, _ = raw[0, start_sec * SFREQ : end_sec * SFREQ]
    data = data.flatten()

    segments = []
    # Ранее шаг был WINDOW_SAMPLES (без перекрытия)
    # Теперь изменен на WINDOW_SAMPLES // 2 (перекрытие 50%), объем данных удваивается
    step = WINDOW_SAMPLES // 2
    for i in range(0, len(data) - WINDOW_SAMPLES, step):
        if len(segments) >= max_segments:
            break
        segments.append(data[i : i + WINDOW_SAMPLES])
    return segments

# Увеличение целевого max_segments, по 150-200 изображений для каждой категории
# Таким образом, в валидационном наборе будет 60-80 изображений, и точность станет более стабильной

print("Анализ EDF файлов и извлечение временных сегментов...")
seizure_segments = []
normal_segments = []

# 1. Извлечение фрагментов с приступами (цель: не менее 50)
for filename, start, end in seizure_records:
    filepath = os.path.join(DATA_DIR, filename)
    if os.path.exists(filepath):
        raw = mne.io.read_raw_edf(filepath, preload=True)
        # Извлечение примерно 10 фрагментов из каждого файла, итого 50 из 5 файлов
        segments = extract_segments(raw, start, end, max_segments=10)
        seizure_segments.extend(segments)

# 2. Извлечение нормальных фрагментов (цель: не менее 50)
for filename in normal_records:
    filepath = os.path.join(DATA_DIR, filename)
    if os.path.exists(filepath):
        raw = mne.io.read_raw_edf(filepath, preload=True)
        # Извлечение фрагментов из средней части нормальных файлов (например, начиная с 1000-й секунды)
        segments = extract_segments(raw, 1000, 1200, max_segments=17)
        normal_segments.extend(segments)

print(f"Извлечение завершено: {len(seizure_segments)} фрагментов с приступами, {len(normal_segments)} нормальных фрагментов")

Анализ EDF файлов и извлечение временных сегментов...


C:\Users\Sunshine\AppData\Local\Temp\ipykernel_36304\3038715444.py:62: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(filepath, preload=True)
C:\Users\Sunshine\AppData\Local\Temp\ipykernel_36304\3038715444.py:62: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(filepath, preload=True)
C:\Users\Sunshine\AppData\Local\Temp\ipykernel_36304\3038715444.py:62: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(filepath, preload=True)
C:\Users\Sunshine\AppData\Local\Temp\ipykernel_36304\3038715444.py:62: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(filepath, preload=True)
C:\Users\Sunshine\AppData\Lo

Извлечение завершено: 50 фрагментов с приступами, 51 нормальных фрагментов


In [50]:
def generate_scalogram(signal, save_path):
    """Вычисляет непрерывное вейвлет-преобразование и сохраняет как изображение"""
    # Установка масштабов вейвлета (от 1 до 64)
    scales = np.arange(1, 65)
    # Использование вейвлета Морле ('morl')
    coefs, freqs = pywt.cwt(signal, scales, 'morl')

    # Взятие абсолютного значения и построение графика
    plt.figure(figsize=(3, 3), dpi=100) # Генерация изображения 300x300
    plt.imshow(abs(coefs), extent=[0, WINDOW_SEC, 1, 64], cmap='jet', aspect='auto', origin='lower')
    plt.axis('off') # Отключение осей для сохранения только признаков изображения
    plt.tight_layout(pad=0)
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
    plt.close() # Освобождение памяти для предотвращения утечек!

print("Генерация частотно-временных диаграмм...")
# Генерация 50 изображений с приступами
for i, seg in enumerate(tqdm(seizure_segments[:50], desc="Seizure Scalograms")):
    generate_scalogram(seg, os.path.join(OUT_DIR_SEIZURE, f"seizure_{i}.png"))

# Генерация 50 нормальных изображений
for i, seg in enumerate(tqdm(normal_segments[:50], desc="Normal Scalograms")):
    generate_scalogram(seg, os.path.join(OUT_DIR_NORMAL, f"normal_{i}.png"))
print("Генерация и сохранение 100 изображений по категориям завершено!")

Генерация частотно-временных диаграмм...


Normal Scalograms: 100%|██████████| 50/50 [00:05<00:00,  8.36it/s]

Генерация и сохранение 100 изображений по категориям завершено!


In [51]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

class EEG_CNN(nn.Module):
    def __init__(self, num_classes=2):
        super(EEG_CNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 224 -> 112

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 112 -> 56

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 56 -> 28
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Предварительная обработка данных
transform = transforms.Compose([
    transforms.Resize((224, 224)), # Немного уменьшенный размер для ускорения обучения и лучшего эффекта
    transforms.RandomHorizontalFlip(), # Случайное горизонтальное отражение
    transforms.RandomRotation(10),     # Случайный поворот на небольшой угол
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Загрузка датасета (PyTorch автоматически устанавливает метки по именам папок seizure / normal)
dataset = datasets.ImageFolder(root='./dataset', transform=transform)

# Разделение на обучающую (80%) и валидационную (20%) выборки
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EEG_CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

print(f"Модель загружена на {device}. Сопоставление классов: {dataset.class_to_idx}")

Модель загружена на cpu. Сопоставление классов: {'normal': 0, 'seizure': 1}


In [52]:
# Инициализация переменных
best_val_acc = 0.0  # Для записи максимальной точности
EPOCHS = 50
SAVE_PATH = 'eeg_best_model.pth' # Имя файла лучшей модели

for epoch in range(EPOCHS):
    # --- Этап обучения ---
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # --- Этап валидации ---
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total
    avg_loss = running_loss / len(train_loader)

    # --- Логика сохранения лучшей модели ---
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        # Сохранение весов модели
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_loss:.4f} | Val Acc: {val_acc:.2f}% (Обнаружена лучшая модель, сохранено!)")
    else:
        print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_loss:.4f} | Val Acc: {val_acc:.2f}%")

print("--------------------------------------------------")
print(f"Обучение завершено! Лучшая модель сохранена в: {SAVE_PATH}")
print(f"Максимальная точность на валидации: {best_val_acc:.2f}%")

Epoch [1/50] Loss: 0.7255 | Val Acc: 40.00% (Обнаружена лучшая модель, сохранено!)
Epoch [2/50] Loss: 0.6957 | Val Acc: 50.00% (Обнаружена лучшая модель, сохранено!)
Epoch [3/50] Loss: 0.6329 | Val Acc: 65.00% (Обнаружена лучшая модель, сохранено!)
Epoch [4/50] Loss: 0.5892 | Val Acc: 75.00% (Обнаружена лучшая модель, сохранено!)
Epoch [5/50] Loss: 0.5451 | Val Acc: 60.00%
Epoch [6/50] Loss: 0.5046 | Val Acc: 65.00%
Epoch [7/50] Loss: 0.4772 | Val Acc: 65.00%
Epoch [8/50] Loss: 0.4511 | Val Acc: 65.00%
Epoch [9/50] Loss: 0.4792 | Val Acc: 70.00%
Epoch [10/50] Loss: 0.4383 | Val Acc: 70.00%
Epoch [11/50] Loss: 0.4601 | Val Acc: 75.00%
Epoch [12/50] Loss: 0.4261 | Val Acc: 70.00%
Epoch [13/50] Loss: 0.4271 | Val Acc: 70.00%
Epoch [14/50] Loss: 0.3906 | Val Acc: 80.00% (Обнаружена лучшая модель, сохранено!)
Epoch [15/50] Loss: 0.4716 | Val Acc: 70.00%
Epoch [16/50] Loss: 0.4257 | Val Acc: 75.00%
Epoch [17/50] Loss: 0.3693 | Val Acc: 80.00%
Epoch [18/50] Loss: 0.3718 | Val Acc: 75.00%
Epoc